<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 지식 증류 · 모델 경량화 · 00. Knowledge Distillation

## Distilling the Knowledge in a Neural Network

- **원 논문:** [Distilling the Knowledge in a Neural Network](https://research.google/pubs/distilling-the-knowledge-in-a-neural-network/)
- **저자 / 발표:** Geoffrey Hinton, Oriol Vinyals, Jeff Dean · NeurIPS 2014 Deep Learning Workshop paper (published 2015) (2015)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** PyTorch tensor와 autograd, softmax·cross entropy, 선형대수 기초
- **로컬 학습 데이터:** `data/field_curriculum/compression_bench.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** temperature로 부드러워진 teacher 분포와 hard label을 함께 사용해 작은 student를 학습한다.

**축소하거나 생략한 부분:** 원 논문은 ensemble과 대형 음성·영상 모델을 다뤘다. 실습은 32차원 합성 분류의 linear student로 dark knowledge만 분리해 본다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (1): temperature softmax | Task 1 | 온도 증가 시 entropy 변화 |
| §2, Eq. (2)–(4): distillation gradient | Task 2 | T²가 포함된 KL loss |
| §2: soft targets + correct labels | Task 3 | KD+CE student 학습 |
| §3.1: specialist/teacher transfer | Task 3 | teacher agreement와 loss plot |

## 미니 재현
정답 class만이 아니라 teacher가 다른 class에 나눠 준 작은 확률, 즉 **dark knowledge**를 student가 배우는지 확인합니다.

## 포트폴리오 해설: 정확도와 효율의 계약

$$\mathcal L=\alpha T^2D_{KL}(p_T^{teacher}\Vert p_T^{student})+(1-\alpha)CE(y,p_1^{student})$$

- **기호 정의:** T는 temperature, α는 soft/hard loss 비율, p_T는 온도 T의 softmax입니다.
- **수식의 역할:** teacher의 class 간 상대 확률인 dark knowledge와 정답 label을 함께 전달합니다.
- **구현 이유:** 정확도만 남기면 경량화의 비용 절감 주장을 검증할 수 없으므로,
  핵심 변환과 품질·비용 측정을 독립 함수와 method로 분리했습니다.
- **Task/코드 대응:** Task P의 논문 전용 class와 `forward`, `compute_loss`,
  `training_step`, `fit_portfolio_model`, `evaluate_portfolio_model`이
  §2, Eq. (1): temperature softmax의 흐름을 명시합니다.
- **입출력 shape:** 특징 [B, 32] → student logit [B, 4], teacher logit [B, 4]
- **평가:** KD loss, test accuracy, teacher/student KL divergence
- **원문 대비 한계:** 원 논문은 ensemble과 대형 음성·영상 모델을 다뤘다. 실습은 32차원 합성 분류의 linear student로 dark
knowledge만 분리해 본다.

경량화 실험에서는 정확도만 보지 않고 parameter·sparsity·bit·latency 중 논문이
주장한 비용을 함께 기록해야 합니다. 아래 코드는 단계별 중간값을 남깁니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** features [B,D] + teacher logits [B,K] → student logits [B,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(ACCELERATOR.summary())


def locate(relative):
    for base in (Path.cwd(), *Path.cwd().parents):
        path = base / relative
        if path.exists():
            return path
    raise FileNotFoundError(relative)


dataset_path = locate("data/field_curriculum/compression_bench.npz")
raw = np.load(dataset_path, allow_pickle=False)
features = torch.from_numpy(raw["features"]).float().to(DEVICE)
labels = torch.from_numpy(raw["labels"]).long().to(DEVICE)
teacher_logits = torch.from_numpy(raw["teacher_logits"]).float().to(DEVICE)
teacher_weights = torch.from_numpy(raw["teacher_weights"]).float().to(DEVICE)
calibration = torch.from_numpy(raw["calibration_activations"]).float().to(DEVICE)
dense_weights = torch.from_numpy(raw["dense_weights"]).float().to(DEVICE)
train_idx = torch.from_numpy(raw["train_idx"]).long().to(DEVICE)
test_idx = torch.from_numpy(raw["test_idx"]).long().to(DEVICE)
images = torch.from_numpy(raw["images"]).float().to(DEVICE)
image_labels = torch.from_numpy(raw["image_labels"]).long().to(DEVICE)
image_train_idx = torch.from_numpy(raw["image_train_idx"]).long().to(DEVICE)
image_test_idx = torch.from_numpy(raw["image_test_idx"]).long().to(DEVICE)
token_embeddings = torch.from_numpy(raw["token_embeddings"]).float().to(DEVICE)
teacher_hidden = torch.from_numpy(raw["teacher_hidden"]).float().to(DEVICE)
teacher_attention = torch.from_numpy(raw["teacher_attention"]).float().to(DEVICE)
candidate_configs = torch.from_numpy(raw["candidate_configs"]).float()
assert features.shape == (512, 32) and teacher_logits.shape == (512, 4)
assert images.shape == (320, 1, 16, 16)
assert teacher_hidden.shape == (256, 6, 24)
print(
    dataset_path.name, "features", tuple(features.shape), "images", tuple(images.shape)
)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (1): temperature softmax
- **이 셀의 책임:** Task 1
- **Tensor shape 계약:** features [B,D] + teacher logits [B,K] → student logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 온도 증가 시 entropy 변화. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def soft_targets(logits, temperature):
    return F.softmax(logits / temperature, dim=-1)


def entropy(prob):
    return -(prob * prob.clamp_min(1e-9).log()).sum(-1).mean()


p1, p4 = soft_targets(teacher_logits, 1.0), soft_targets(teacher_logits, 4.0)
assert torch.allclose(p4.sum(-1), torch.ones(len(p4)), atol=1e-6)
assert entropy(p4) > entropy(p1)
print("entropy T=1/T=4:", float(entropy(p1)), float(entropy(p4)))

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §2, Eq. (2)–(4): distillation gradient / §2: soft targets + correct labels
- **이 셀의 책임:** Task 2 / Task 3
- **Tensor shape 계약:** features [B,D] + teacher logits [B,K] → student logits [B,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** T²가 포함된 KL loss / KD+CE student 학습. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def kd_loss(student, teacher, target, temperature=4.0, alpha=0.7):
    soft = (
        F.kl_div(
            F.log_softmax(student / temperature, dim=-1),
            F.softmax(teacher / temperature, dim=-1),
            reduction="batchmean",
        )
        * temperature**2
    )
    hard = F.cross_entropy(student, target)
    return alpha * soft + (1 - alpha) * hard


probe = torch.zeros(8, 4, requires_grad=True)
loss_probe = kd_loss(probe, teacher_logits[:8], labels[:8])
loss_probe.backward()
assert torch.isfinite(loss_probe) and probe.grad.abs().sum() > 0

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §2: soft targets + correct labels / §3.1: specialist/teacher transfer
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** features [B,D] + teacher logits [B,K] → student logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** KD+CE student 학습 / teacher agreement와 loss plot. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
student = nn.Linear(32, 4)
optimizer = torch.optim.Adam(student.parameters(), lr=0.04)
losses = []
before = F.kl_div(
    F.log_softmax(student(features[test_idx]), -1),
    F.softmax(teacher_logits[test_idx], -1),
    reduction="batchmean",
).item()
for _ in range(60):
    optimizer.zero_grad()
    logits = student(features[train_idx])
    loss = kd_loss(logits, teacher_logits[train_idx], labels[train_idx])
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))
with torch.no_grad():
    predicted = student(features[test_idx])
    after = F.kl_div(
        F.log_softmax(predicted, -1),
        F.softmax(teacher_logits[test_idx], -1),
        reduction="batchmean",
    ).item()
    accuracy = (predicted.argmax(-1) == labels[test_idx]).float().mean().item()
assert losses[-1] < losses[0] and after < before and accuracy > 0.85
plt.plot(losses)
plt.title(f"KD student: acc={accuracy:.1%}")
plt.xlabel("step")
plt.ylabel("loss")
plt.show()
print(f"teacher KL {before:.3f}→{after:.3f}, accuracy={accuracy:.1%}")

### Task P1 · 압축 대상과 핵심 모델

tensor shape와 비용 단위를 먼저 확인한 뒤 논문 전용 class의 `forward`와
`compute_loss`에 정확도·표현 보존 계약을 명시합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (1): temperature softmax / §2, Eq. (2)–(4): distillation gradient / §2: soft targets + correct labels / §3.1: specialist/teacher transfer
- **이 셀의 책임:** Task 1 / Task 2 / Task 3
- **Tensor shape 계약:** features [B,D] + teacher logits [B,K] → student logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 온도 증가 시 entropy 변화 / T²가 포함된 KL loss / KD+CE student 학습 / teacher agreement와 loss plot. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 16
    temperature: float = 3.0
    soft_weight: float = 0.7
    learning_rate: float = 0.03
    steps: int = 18


def prepare_batch() -> tuple[Tensor, Tensor, Tensor]:
    return features[train_idx], labels[train_idx], teacher_logits[train_idx]


class KnowledgeDistillationPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.student = nn.Sequential(
            nn.Linear(features.shape[1], config.hidden_dim),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, teacher_logits.shape[1]),
        )

    def forward(self, inputs: Tensor) -> Tensor:
        return self.student(inputs)

    def compute_loss(
        self,
        student_logits: Tensor,
        targets: Tensor,
        teacher: Tensor,
    ) -> Tensor:
        temperature = self.config.temperature
        soft = (
            F.kl_div(
                F.log_softmax(student_logits / temperature, dim=1),
                F.softmax(teacher / temperature, dim=1),
                reduction="batchmean",
            )
            * temperature**2
        )
        hard = F.cross_entropy(student_logits, targets)
        weight = self.config.soft_weight
        return weight * soft + (1.0 - weight) * hard

    def training_step(self, inputs: Tensor, targets: Tensor, teacher: Tensor) -> Tensor:
        return self.compute_loss(self(inputs), targets, teacher)

### Task P2 · 학습·pruning·distillation update

`fit_portfolio_model`에서 optimizer, mask/EMA/quantization 경계와 update 순서를
드러냅니다. 비용 절감이 학습 경로에 미치는 영향을 함께 읽으세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §2, Eq. (2)–(4): distillation gradient / §2: soft targets + correct labels
- **이 셀의 책임:** Task 2 / Task 3
- **Tensor shape 계약:** features [B,D] + teacher logits [B,K] → student logits [B,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** T²가 포함된 KL loss / KD+CE student 학습. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: KnowledgeDistillationPortfolioModel) -> list[float]:
    inputs, targets, teacher = prepare_batch()
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(inputs, targets, teacher)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 품질과 비용을 함께 평가

accuracy 또는 reconstruction 품질과 parameter·sparsity·bit·latency를 별도로
기록합니다. 로컬 지표는 원문 하드웨어 benchmark의 대체값이 아닙니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §3.1: specialist/teacher transfer
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** features [B,D] + teacher logits [B,K] → student logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** teacher agreement와 loss plot. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: KnowledgeDistillationPortfolioModel,
) -> dict[str, float]:
    with torch.no_grad():
        logits = model(features[test_idx])
        accuracy = (logits.argmax(1) == labels[test_idx]).float().mean()
        divergence = F.kl_div(
            F.log_softmax(logits, dim=1),
            F.softmax(teacher_logits[test_idx], dim=1),
            reduction="batchmean",
        )
    return {
        "test_accuracy": float(accuracy.cpu()),
        "teacher_kl": float(divergence.cpu()),
    }


portfolio_model = KnowledgeDistillationPortfolioModel(PortfolioConfig())
portfolio_history = fit_portfolio_model(portfolio_model)
portfolio_metrics = evaluate_portfolio_model(portfolio_model)
assert np.isfinite(portfolio_history).all()
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 원 논문은 ensemble과 대형 음성·영상 모델을 다뤘다. 실습은 32차원 합성 분류의 linear student로 dark knowledge만 분리해 본다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.